# SlowFast Networks for Video Recognition (2019)

**Authors:** Christoph Feichtenhofer, Haoqi Fan, Jitendra Malik, Kaiming He  
**Year:** 2019

---

# Images vs Videos

Images have the following characteristics:

- The spatial domain `(h, w)` is:

    - **Symmetric**: horizontal and vertical patterns are equally important.  
      This is why 2D convolutions typically use kernels such as `3×3`, rather than unusual shapes like `3×15` or `1×20`.

      It is assumed that horizontal and vertical patterns are equally relevant.

    - **Isotropic**: statistical properties are similar in all directions.
        - vertical edges appear frequently
        - horizontal edges also appear frequently
        - diagonal patterns appear as well

      There is no privileged orientation.

    - **Shift-invariant**: an object remains the same even if its position changes.
        - a dog in the top-left corner
        - a dog in the bottom-right corner

      are still recognized as a dog.

      CNNs exploit this property by sharing filters across the entire image.

- However, in videos, time comes into play, and time is **NOT isotropic**.

    Time has:
    - direction
    - causality
    - order

    Example:
    - a person falling
    - a person standing up

    If time is reversed:

    it is no longer the same action.

---

# Architecture

The architecture consists of two pathways:
- a **Slow pathway**
- a **Fast pathway**

The Slow pathway captures semantic information, while the Fast pathway captures motion dynamics.

---

## Slow Pathway

The Slow pathway can be any convolutional model that processes a video clip as a spatiotemporal volume. Therefore, 3D operations such as convolutions and pooling layers are required.

The pathway uses a temporal frame stride $\tau$ (frame skip), meaning that only one out of every $\tau$ frames is sampled. The total number of sampled frames is denoted by $T$.

For example, if $\tau = 16$ and the original clip contains 64 frames, then:

```math
T = \frac{64}{16} = 4
```

Thus, the model processes 4 sampled frames, while the original clip length is:

```math
\tau \times T = 64
```

---

## Fast Pathway

Similar to the Slow pathway, it is also a convolutional model, with the following considerations:

### High Frame Rate

The Fast pathway uses a smaller temporal stride, $\tau / \alpha$, with $\alpha > 1$, allowing it to process more frames than the Slow pathway.

Here, $\alpha$ represents the frame rate ratio between the Fast and Slow pathways. As a result, the Fast pathway processes $\alpha$ times more frames than the Slow pathway, for a total of $\alpha T$ frames.

A typical value used in the paper is:

```math
\alpha = 8
```

---

### High Temporal Resolution Features

The Fast pathway does not perform temporal downsampling (such as temporal pooling or time-strided convolutions). As a result, it preserves temporal fidelity as much as possible throughout the network.

---

### Low Channel Capacity

Since the Fast pathway is designed to model motion, it processes more frames (higher temporal resolution). However, it uses fewer channels, making it computationally lightweight despite processing more frames.

To achieve this, a factor of $\beta$ $(\beta < 1)$ is used to reduce the number of channels. A typical value used in the paper is:

```math
\beta = \frac{1}{8}
```

For example, if the Slow pathway has 256 channels, then the Fast pathway will have:

```math
256 \times \frac{1}{8} = 32
```

channels.

---

## Lateral Connections

Lateral connections are a popular technique for merging different levels of spatial resolution and semantic information. In SlowFast, a lateral connection is attached between the two pathways at multiple stages of the network.

When using a ResNet backbone, the lateral connections are added after:
- `pool1`
- `res2`
- `res3`
- `res4`

Since the two pathways operate at different temporal resolutions, their feature tensors have different temporal dimensions. Therefore, the lateral connections perform a transformation to align the temporal dimensions before fusion.

**The paper uses unidirectional lateral connections, where features from the Fast pathway are fused into the Slow pathway.**

---

## Output

Finally:

- a global average pooling is performed on each pathway’s output,
- the pathway features are concatenated,
- a fully connected layer is used,
- and a softmax layer produces the final prediction.

<div>
    <img src='../../images/SlowFast.png'>
</div>

---

# Network Instantiation

For the **Slow pathway**, `res4` and `res5` use **non-degenerate** temporal convolutions (temporal kernel size > 1). This means they use regular 3D convolutions.

However, the previous stages use **degenerate** temporal convolutions (temporal kernel size = 1), which behave similarly to 2D convolutions applied independently over each frame.

In other words, a regular 3D convolution may use a kernel such as:

```math
3 \times 3 \times 3
```

while a degenerate temporal convolution uses:

```math
1 \times 3 \times 3
```

We call it **non-degenerate** because, in a regular 3D convolution, the temporal kernel size is used to process multiple consecutive frames (each one with 3 RGB channels) simultaneously. Since multiple frames are involved, temporal information exists between them (i.e., motion), allowing the convolution to mix temporal information across frames and learn motion dynamics.

However, when the temporal kernel is **degenerate**, the Conv3D operation only processes one frame at a time (still with its 3 RGB channels). Since only a single frame is used, no temporal relationship or motion exists between frames, so no temporal information is mixed.

For the **Fast pathway**:
- $\alpha = 8$
- $\beta = \frac{1}{8}$

The Fast pathway has non-degenerate temporal convolutions in every block.

---

# Lateral Connection Transformations

The **lateral connections** are fused from the Fast pathway into the Slow pathway.

- The Slow pathway output has the shape:

```math
(T, S^2, C)
```

where:
- $S^2$ represents the spatial image dimensions $(h \times w)$,
- for the experiments both are set to 224,
- and $C$ represents the number of channels.

- The Fast pathway output has the shape:

```math
(\alpha T, S^2, \beta C)
```

Therefore, the Fast pathway features must be transformed to become compatible with the Slow pathway before fusion.

---

## 1. Time-to-channel

In the Time-to-channel transformation, the temporal dimension is reshaped into the channel dimension.

The Fast pathway feature tensor:

```math
(\alpha T, S^2, \beta C)
```

is transformed into:

```math
(T, S^2, \alpha \beta C)
```

This means that all $\alpha$ frames are packed into the channels of a single frame.

> Although the number of channels increases, this is acceptable because convolutional layers can later learn how to combine and reduce channel information. In CNNs, channels mainly represent feature dimensions, and later convolutions can freely transform them into new feature maps.

---

## 2. Time-strided Sampling

In the Time-strided sampling transformation, the Fast pathway features are temporally downsampled by simply selecting one out of every $\alpha$ frames.

The Fast pathway feature tensor:

```math
(\alpha T, S^2, \beta C)
```

is transformed into:

```math
(T, S^2, \beta C)
```

by sampling every $\alpha$ temporal positions.

For example, if:

```math
\alpha = 8
```

then only one frame out of every 8 frames is retained.

This method is computationally simple, but some temporal information and fine motion details may be lost during sampling.

---

## 3. Time-strided Convolution

Another proposed transformation uses a temporal 3D convolution to reduce the temporal dimension of the Fast pathway before fusion.

The output size of a convolution is computed as:

```math
O = \left\lfloor \frac{I + 2P - K}{S} \right\rfloor + 1
```

where:
- $I$ = input size
- $P$ = padding
- $K$ = kernel size
- $S$ = stride
- $O$ = output size

Applied to the temporal dimension:

```math
T_{out} = \left\lfloor \frac{\alpha T + 2P - K}{\alpha} \right\rfloor + 1
```

For example, using:
- temporal kernel size = 5
- temporal stride = $\alpha$
- padding = 2

the temporal dimension is reduced from:

```math
\alpha T \rightarrow T
```

allowing the Fast pathway features to match the temporal size of the Slow pathway before fusion.

## Network parameters

The network is the next one:

<div>
    <img src='../../images/SlowFastParams.png'>
</div>

The table is interpretated as follows:

> For Conv3D the spatial stride is the regular stride which affects $h$ and $w$.

| Stage          | Slow Pathway                                                           | Fast Pathway                                       | Output Size                         |
| -------------- | ---------------------------------------------------------------------- | -------------------------------------------------- | ----------------------------------- |
| **Input clip** | Raw video clip                                                         | Raw video clip                                     | `64 × 224 × 224`                    |
| **Data layer** | Temporal sampling: stride `τ = 16` → `4` frames                        | Temporal sampling: stride `2` → `32` frames        | Slow: `4 × 224²`  Fast: `32 × 224²` |
| **conv1**      | Conv3D `1×7×7`, 64 channels, spatial stride `2`                        | Conv3D `5×7×7`, 8 channels, spatial stride `2`     | Slow: `4 × 112²`  Fast: `32 × 112²` |
| **pool1**      | MaxPool `1×3×3`, spatial stride `2`                                    | MaxPool `1×3×3`, spatial stride `2`                | Slow: `4 × 56²`  Fast: `32 × 56²`   |
| **res2**       | Residual blocks ×3  Kernels mostly `1×1×1`, `1×3×3`                    | Residual blocks ×3  Includes temporal conv `3×1×1` | Slow: `4 × 56²`  Fast: `32 × 56²`   |
| **res3**       | Residual blocks ×4  Still mostly degenerate temporal convs             | Residual blocks ×4  Temporal convs maintained      | Slow: `4 × 28²`  Fast: `32 × 28²`   |
| **res4**       | Residual blocks ×6  Introduces non-degenerate temporal convs (`3×1×1`) | Residual blocks ×6  Temporal convs maintained      | Slow: `4 × 14²`  Fast: `32 × 14²`   |
| **res5**       | Residual blocks ×3  Non-degenerate temporal convs                      | Residual blocks ×3  Non-degenerate temporal convs  | Slow: `4 × 7²`  Fast: `32 × 7²`     |
| **Output**     | Global Average Pool                                                    | Global Average Pool                                | Concatenate → FC → Softmax          |